In [1]:
import pandas as pd
import subprocess
import os

# Input CSV file
fin = 'test_numbers.csv'

# Create output directories if they don't exist
os.makedirs('./output_xml_sh', exist_ok=True)
os.makedirs('./skymaps', exist_ok=True)
os.makedirs('./sm_tmp', exist_ok=True)

# Read CSV into a DataFrame
df = pd.read_csv(fin)
print(df.columns)

if df.columns[0].startswith("Unnamed"):
    df.rename(columns={df.columns[0]: "event"}, inplace=True)

# Loop over events (rows)
for _, row in df.iterrows():
    # Extract variables
    event = row['event']
    geocent_time = float(row['geocent_time'])
    luminosity_distance = float(row['luminosity_distance']) #from halos
    m1 = float(row['mass_1_source']) #distribution
    m2 = float(row['mass_2_source']) #distribution from mass ratio q=m2/m1
    dec = float(row['dec']) #from halos
    ra = float(row['ra']) #from halos
    theta_jn = float(row['theta_jn']) #random
    psi = float(row['psi']) #random
    phase = float(row['phase']) #random
    a_1 = float(row['a_1']) #distribution
    a_2 = float(row['a_2']) #distribution

    print(f"{event},{geocent_time:.3f}")

    # GPS times
    time = int(geocent_time)
    gps_start = time - 210
    gps_end = time + 50

    # Convert radians to degrees
    theta_jn_deg = theta_jn * 57.3
    phase_deg = phase * 57.3
    psi_deg = psi * 57.3

    # Spin
    spin1_max = round(a_1 + 0.001, 3)
    spin2_max = round(a_2 + 0.001, 3)

    # Distances
    luminosity_distance_km = int(luminosity_distance * 1000)
    luminosity_distance_max = luminosity_distance_km + 1

    # RA range check
    ra_deg = ra * 57.3
    if int(ra_deg) > 180:
        ra_deg -= 360

    # DEC range check
    dec_deg = dec * 57.3
    if int(dec_deg) > 90:
        dec_deg -= 360

    # Run lalapps_inspinj
    lal_cmd = [
        "lalapps_inspinj",
        "--gps-start-time", str(gps_start),
        "--gps-end-time", str(gps_end),
        "--time-step", "200",
        "--m-distr", "fixMasses",
        "--fixed-mass1", str(m1),
        "--fixed-mass2", str(m2),
        "--i-distr", "fixed",
        "--fixed-inc", str(theta_jn_deg),
        "--polarization", str(psi_deg),
        "--fixed-coa-phase", str(phase_deg),
        "--l-distr", "fixed",
        "--longitude", str(ra_deg),
        "--latitude", str(dec_deg),
        "--waveform", "SEOBNRv4PHM",
        "--f-lower", "10",
        "--taper-injection", "startend",
        "--d-distr", "uniform",
        "--min-distance", str(luminosity_distance_km),
        "--max-distance", str(luminosity_distance_max),
        "--enable-spin",
        "--min-spin1", str(a_1),
        "--max-spin1", str(spin1_max),
        "--min-spin2", str(a_2),
        "--max-spin2", str(spin2_max),
        "--band-pass-injection"
    ]
    print(f"Running lalapps_inspinj for {event}...")
    subprocess.run(lal_cmd, check=True)

    # Rename XML output
    lal_output_files = [f for f in os.listdir('.') if f.startswith("HL-INJECTIONS_1") and f.endswith(".xml")]
    if not lal_output_files:
        raise FileNotFoundError(f"No lalapps XML output found for {event}")
    lal_output = lal_output_files[0]
    lal_new_name = f'./output_xml_sh/{event}_10s_before_lalapp.xml'
    os.rename(lal_output, lal_new_name)

    # Run bayestar
    coinc_name = f'./output_xml_sh/bayestar_coinc_{event}_10s_before_lalapp.xml'
    bayestar_realize_cmd = [
        "bayestar-realize-coincs",
        "-o", coinc_name,
        lal_new_name,
        "--reference-psd", "./psd/test.xml",
        "--detector", "H1", "L1", "V1", #L1:aLIGOaLIGODesignSensitivityT1800044 #H1:aLIGOaLIGODesignSensitivityT1800044 #V1:AdVDesignSensitivityP1200087
        "--measurement-error", "gaussian-noise",
        "--snr-threshold", "4.0",
        "--net-snr-threshold", "8.0",
        "--min-triggers", "2",
        "--keep-subthreshold"
    ]
    try:
        subprocess.run(bayestar_realize_cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"Bayestar failed for {event} with exit code {e.returncode}")
        print("stdout:", e.stdout)
        print("stderr:", e.stderr)
        continue  # skip to next event

    os.environ["OMP_NUM_THREADS"] = "4"
    bayestar_localize_cmd = [
        "bayestar-localize-coincs",
        coinc_name,
        "--output", "./sm_tmp",
        "--waveform", "SEOBNRv4PHM",
        "--f-low", "10",
    ]
    subprocess.run(bayestar_localize_cmd, check=True)

    # Move final FITS file
    fits_name = f'./skymaps/{event}_test.fits'
    fits_files = [f for f in os.listdir('./sm_tmp') if f.endswith('.fits')]

    if fits_files:
        fits_input = os.path.join('./sm_tmp', fits_files[0])
        os.rename(fits_input, fits_name)
        print(f"Saved skymap: {fits_name}")
    else:
        print(f"No FITS file found for {event}")

# Run final Python script
subprocess.run(["python", "read_all_skymaps.py"], check=True)

Index(['Unnamed: 0', 'geocent_time', 'luminosity_distance', 'mass_1_source',
       'mass_2_source', 'dec', 'ra', 'theta_jn', 'psi', 'phase', 'a_1', 'a_2'],
      dtype='object')
GW1,1215324652.000
Running lalapps_inspinj for GW1...


2026-04-09 20:15:44,714 INFO Using 4 OpenMP thread(s)
2026-04-09 20:15:44,714 INFO ./output_xml_sh/bayestar_coinc_GW1_10s_before_lalapp.xml:reading input files
2026-04-09 20:15:44,739 WARNING Using anti-FINDCHIRP phase convention; inverting phases. This is currently the default and it is appropriate for gstlal and MBTA but not pycbc as of observing run 1 ("O1"). The default setting is likely to change in the future.
2026-04-09 20:15:44,742 INFO 0:computing sky map
2026-04-09 20:15:44,783 WARNING Truncating PSD at 0.95 of maximum frequency to suppress rolloff artifacts. This option may be removed in the future.
2026-04-09 20:15:44,785 WARNING Truncating PSD at 0.95 of maximum frequency to suppress rolloff artifacts. This option may be removed in the future.
2026-04-09 20:15:44,786 WARNING Truncating PSD at 0.95 of maximum frequency to suppress rolloff artifacts. This option may be removed in the future.
2026-04-09 20:15:44,788 INFO Selected template: SEOBNRv4PHM
2026-04-09 20:15:47,179 

Saved skymap: ./skymaps/GW1_test.fits
GW2,1215135456.000
Running lalapps_inspinj for GW2...


2026-04-09 20:17:27,085 INFO Using 4 OpenMP thread(s)
2026-04-09 20:17:27,086 INFO ./output_xml_sh/bayestar_coinc_GW2_10s_before_lalapp.xml:reading input files
2026-04-09 20:17:27,116 WARNING Using anti-FINDCHIRP phase convention; inverting phases. This is currently the default and it is appropriate for gstlal and MBTA but not pycbc as of observing run 1 ("O1"). The default setting is likely to change in the future.
2026-04-09 20:17:27,117 INFO 0:computing sky map
2026-04-09 20:17:27,184 WARNING Truncating PSD at 0.95 of maximum frequency to suppress rolloff artifacts. This option may be removed in the future.
2026-04-09 20:17:27,185 WARNING Truncating PSD at 0.95 of maximum frequency to suppress rolloff artifacts. This option may be removed in the future.
2026-04-09 20:17:27,186 WARNING Truncating PSD at 0.95 of maximum frequency to suppress rolloff artifacts. This option may be removed in the future.
2026-04-09 20:17:27,187 INFO Selected template: SEOBNRv4PHM
2026-04-09 20:17:32,458 

Saved skymap: ./skymaps/GW2_test.fits
GW3,1246835424.000
Running lalapps_inspinj for GW3...


...

2026-04-09 20:25:52,650 INFO Using 4 OpenMP thread(s)
2026-04-09 20:25:52,650 INFO ./output_xml_sh/bayestar_coinc_GW9_10s_before_lalapp.xml:reading input files


Saved skymap: ./skymaps/GW9_test.fits
GW10,1235248652.000
Running lalapps_inspinj for GW10...


2026-04-09 20:25:59,327 INFO Using 4 OpenMP thread(s)
2026-04-09 20:25:59,328 INFO ./output_xml_sh/bayestar_coinc_GW10_10s_before_lalapp.xml:reading input files
2026-04-09 20:25:59,348 WARNING Using anti-FINDCHIRP phase convention; inverting phases. This is currently the default and it is appropriate for gstlal and MBTA but not pycbc as of observing run 1 ("O1"). The default setting is likely to change in the future.
2026-04-09 20:25:59,348 INFO 0:computing sky map
2026-04-09 20:25:59,380 WARNING Truncating PSD at 0.95 of maximum frequency to suppress rolloff artifacts. This option may be removed in the future.
2026-04-09 20:25:59,382 WARNING Truncating PSD at 0.95 of maximum frequency to suppress rolloff artifacts. This option may be removed in the future.
2026-04-09 20:25:59,383 WARNING Truncating PSD at 0.95 of maximum frequency to suppress rolloff artifacts. This option may be removed in the future.
2026-04-09 20:25:59,384 INFO Selected template: SEOBNRv4PHM
2026-04-09 20:26:00,973

Saved skymap: ./skymaps/GW10_test.fits
['GW190513_205428_test.fits', 'GW3_test.fits', 'GW4_test.fits', 'GW190828_065509_test.fits', 'GW190512_180714_test.fits', 'GW190720_000836.fits', 'GW190828_063405.fits', 'GW190929_012149_test.fits', 'GW190513_205428.fits', 'GW190602_175927_test.fits', 'GW190412_test.fits', 'GW190924_021846_test.fits', 'GW190512_180714.fits', 'GW190602_175927.fits', 'GW190929_012149.fits', 'GW190828_065509.fits', 'GW190720_000836_test.fits', 'GW5_test.fits', 'GW190915_235702.fits', 'GW190728_064510.fits', 'GW8_test.fits', 'GW190517_055101_test.fits', 'fit files', 'GW9_test.fits', 'GW190413_052954.fits', 'GW190828_063405_test.fits', 'GW190408_181802_test.fits', 'GW190408_181802.fits', 'GW190706_222641_test.fits', 'GW190413_052954_test.fits', 'GW7_test.fits', 'GW6_test.fits', 'GW190728_064510_test.fits', '.ipynb_checkpoints', 'GW1_test.fits', 'GW190915_235702_test.fits', 'GW190412.fits', 'GW190701_203306.fits', 'GW190519_153544.fits', 'GW2_test.fits', 'GW190924_02184

CompletedProcess(args=['python', 'read_all_skymaps.py'], returncode=0)